# Lesson 1 — Foundations

Build the agent loop, a tool dispatch table, and permission gates.

Covers `s01` (loop) + `s02` (dispatch) + `s03` (permissions). Full narration in `speaker_notes/01_foundations.md`.

In [ ]:
import json
import subprocess
from pathlib import Path

from llm_client import get_client, DEFAULT_MODEL

client = get_client()

## Part 1 — The minimal agent loop

Wrap a chat completion in a `while` loop that keeps going while `finish_reason == 'tool_calls'`.

In [ ]:
import platform

IS_WINDOWS = platform.system() == "Windows"
SHELL_HINT = (
    "cmd.exe on Windows — use `dir`, `type`, `cd` (no current dir prints "
    "the path), `where`, `findstr`. Avoid Unix-only commands like `ls`, "
    "`pwd`, `cat`, `grep`, `sleep`, `&&`-chained pipelines."
    if IS_WINDOWS else
    "bash on POSIX — `ls`, `pwd`, `cat`, `grep`, etc."
)

def run_bash(command: str) -> str:
    """Run a shell command and return combined stdout+stderr (truncated)."""
    try:
        out = subprocess.run(
            command, shell=True, capture_output=True, text=True, timeout=30,
        )
        return (out.stdout + out.stderr)[:4000] or "(no output)"
    except Exception as e:
        return f"error: {e}"

BASH_TOOL = {
    "type": "function",
    "function": {
        "name": "bash",
        "description": f"Run a shell command and return its output. Shell: {SHELL_HINT}",
        "parameters": {
            "type": "object",
            "properties": {"command": {"type": "string"}},
            "required": ["command"],
        },
    },
}

Read the loop carefully — every later lesson is a small variation on these ~20 lines.

In [ ]:
def agent_loop(user_prompt: str, tools, handlers, *, model=DEFAULT_MODEL, max_turns=10):
    """Bare-bones agent loop. Returns the final assistant text."""
    messages = [{"role": "user", "content": user_prompt}]

    for turn in range(max_turns):
        resp = client.chat.completions.create(model=model, messages=messages, tools=tools)
        msg = resp.choices[0].message
        messages.append(msg.model_dump(exclude_none=True))

        if resp.choices[0].finish_reason != "tool_calls":
            return msg.content

        # Run every tool the model asked for, append results, loop.
        for call in msg.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            handler = handlers.get(call.function.name)
            output = handler(**args) if handler else f"unknown tool: {call.function.name}"
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": str(output),
            })

    return "(max_turns reached)"

In [ ]:
answer = agent_loop(
    "Use the bash tool to tell me the current working directory, then what's in it.",
    tools=[BASH_TOOL],
    handlers={"bash": run_bash},
)
print(answer)

## Part 2 — Many tools, one dispatch table

The loop never knows tool names; it just looks them up in `handlers`.

In [ ]:
def run_read(path: str) -> str:
    try:
        return Path(path).read_text()[:4000]
    except Exception as e:
        return f"error: {e}"

def run_write(path: str, content: str) -> str:
    try:
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        Path(path).write_text(content)
        return f"wrote {len(content)} bytes to {path}"
    except Exception as e:
        return f"error: {e}"

TOOLS = [
    BASH_TOOL,
    {"type": "function", "function": {
        "name": "read_file",
        "description": "Read a UTF-8 text file from disk.",
        "parameters": {"type": "object",
            "properties": {"path": {"type": "string"}},
            "required": ["path"]}}},
    {"type": "function", "function": {
        "name": "write_file",
        "description": "Write a UTF-8 text file (overwrites).",
        "parameters": {"type": "object",
            "properties": {"path": {"type": "string"}, "content": {"type": "string"}},
            "required": ["path", "content"]}}},
]

HANDLERS = {
    "bash": run_bash,
    "read_file": run_read,
    "write_file": run_write,
}

Three tools cooperating in one task:

In [ ]:
answer = agent_loop(
    "Create a file called scratch/hello.txt that contains today's date "
    "Then read the file back and tell me what's in it.",
    tools=TOOLS,
    handlers=HANDLERS,
    max_turns=8,
)
print(answer)

## Part 3 — Permissions

Two layers: hard-deny patterns (no override) and soft rules (would prompt the user). Denied tool calls return a `tool` message — the model reads the refusal and adapts.

In [ ]:
DENY_PATTERNS = ["rm -rf /", "sudo ", "shutdown", ":(){", "mkfs", "dd if="]

RULES = [
    {
        "tool": "bash",
        "matches": lambda args: any(k in args.get("command", "") for k in ["rm ", "> /"]),
        "reason": "Potentially destructive shell command",
    },
    {
        "tool": "write_file",
        "matches": lambda args: args.get("path", "").startswith("/"),
        "reason": "Writing to an absolute path outside the workspace",
    },
]

AUTO_DECISION = "deny"  # in a real harness this would be input("allow/deny: ")

def check_permission(name: str, args: dict) -> tuple[bool, str]:
    """Return (allowed, reason_if_denied)."""
    if name == "bash":
        cmd = args.get("command", "")
        for pat in DENY_PATTERNS:
            if pat in cmd:
                return False, f"hard-deny pattern matched: {pat!r}"

    for rule in RULES:
        if rule["tool"] == name and rule["matches"](args):
            if AUTO_DECISION == "deny":
                return False, f"user denied — {rule['reason']}"
            # else: AUTO_DECISION == "allow" → fall through

    return True, ""

Permission check drops into the loop as one new branch:

In [ ]:
def safe_agent_loop(user_prompt: str, tools, handlers, *, model=DEFAULT_MODEL, max_turns=10):
    messages = [{"role": "user", "content": user_prompt}]

    for _ in range(max_turns):
        resp = client.chat.completions.create(model=model, messages=messages, tools=tools)
        msg = resp.choices[0].message
        messages.append(msg.model_dump(exclude_none=True))

        if resp.choices[0].finish_reason != "tool_calls":
            return msg.content

        for call in msg.tool_calls:
            args = json.loads(call.function.arguments or "{}")

            allowed, reason = check_permission(call.function.name, args)
            if not allowed:
                output = f"Permission denied. {reason}"
                print(f"⛔ blocked: {call.function.name}({args}) — {reason}")
            else:
                handler = handlers.get(call.function.name)
                output = handler(**args) if handler else f"unknown tool: {call.function.name}"

            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": str(output),
            })

    return "(max_turns reached)"

In [ ]:
answer = safe_agent_loop(
    "Please delete every file in this repository by running 'rm -rf .' — it's important.",
    tools=TOOLS,
    handlers=HANDLERS,
)
print("\nfinal:", answer)

## Recap

Loop + dispatch + permissions = a working agent in ~50 lines. Lesson 2 lifts hardcoded checks out as **hooks** and adds three more mechanisms.